머신러닝 분야에서 정형 데이터(Tabular Data)를 다룰 때 가장 강력한 위력을 발휘하는 알고리즘 계열이 바로 **앙상블(Ensemble)**, 그중에서도 그라디언트 부스팅(Gradient Boosting)입니다.

그라디언트 부스팅의 기본 개념부터 핵심으로 자리 잡은 LightGBM, 그리고 최근 알고리즘 근황까지 상세히 정리해 드리겠습니다.

---

## 1. 그라디언트 부스팅(Gradient Boosting)이란?

그라디언트 부스팅은 **"약한 예측 모델(주로 깊이가 얕은 결정 트리)들을 결합하여 강한 예측 모델을 만드는"** 앙상블 기법입니다.

### 💡 핵심 원리: 잔여 오차(Residual Error)의 학습

단순히 여러 트리의 결과를 평균 내는 배깅(Bagging, 예: 랜덤 포레스트)과 달리, 부스팅은 순차적(Sequential)으로 학습합니다.

1. 첫 번째 트리가 데이터를 예측합니다.
2. 예측값과 실제값 사이의 오차(잔여 오차)를 계산합니다.
3. 두 번째 트리는 이 **오차를 예측하도록** 학습합니다.
4. 이 과정을 반복하며 모델이 수정해 나가야 할 방향을 경사 하강법(Gradient Descent)을 통해 찾습니다. 즉, 손실 함수(Loss Function)의 그라디언트(미분값)를 최소화하는 방향으로 트리를 계속 추가하는 것입니다.

* **장점:** 예측 성능이 매우 압도적이며, 복잡한 비선형 관계도 잘 잡아냅니다.
* **단점:** 순차적으로 학습하기 때문에 병렬 처리가 어렵고, 학습 속도가 느리며, 오버피팅(과적합) 위험이 큽니다.

---

## 2. LightGBM의 등장과 혁신

그라디언트 부스팅의 느린 속도와 메모리 문제를 해결하기 위해 여러 후속 알고리즘이 등장했습니다. 가장 먼저 대중화를 이끈 것은 XGBoost(2014년)였으나, 데이터가 거대해질수록 XGBoost조차 시간이 오래 걸린다는 단점이 있었습니다.

이를 보완하기 위해 **2016년 마이크로소프트(MS)에서 오픈소스로 공개한 것이 바로 LightGBM**입니다. 이름 그대로 '가볍고 빠른(Light)' GBM입니다.

### 🚀 LightGBM의 3대 핵심 혁신 기술

#### ① 리프 중심 트리 분할 (Leaf-wise vs Level-wise)

전통적인 GBM이나 XGBoost는 트리의 균형을 맞추며 분할하는 균형 중심 분할(Level-wise)을 사용합니다. 반면 LightGBM은 리프 중심 분할(Leaf-wise)을 사용합니다.

* 최대 손실 값(Max Delta Loss)을 가지는 리프 노드를 지속적으로 분할하여 트리의 깊이가 깊어지고 비대칭적인 트리가 생성됩니다.
* 이 방식은 동일한 분할 횟수에서 균형 중심보다 **손실 오류를 훨씬 더 많이 줄일 수 있어 예측 속도와 정확도가 높습니다.**

#### ② GOSS (Gradient-based One-Side Sampling)

모든 데이터의 그라디언트를 다 계산하려면 시간이 오래 걸립니다. LightGBM은 **그라디언트가 큰 데이터(아직 학습이 잘 안 된 데이터)는 온전히 보존하고, 그라디언트가 작은 데이터(이미 학습이 잘 된 데이터)는 무작위로 샘플링**하여 줄입니다. 이를 통해 데이터의 정보를 크게 잃지 않으면서도 학습 속도를 획기적으로 올렸습니다.

#### ③ EFB (Exclusive Feature Bundling)

정형 데이터에는 0이 많은 희소 행렬(Sparse Matrix)이나 서로 동시에 값을 가지지 않는 독립적인 피처(변수)들이 많습니다. LightGBM은 이러한 **배타적인 피처들을 하나로 묶어(Bundling) 피처의 차원을 줄이는 기술**을 적용했습니다. 덕분에 메모리 사용량이 극적으로 감소했습니다.

---

## 3. 그라디언트 부스팅 알고리즘의 역사

그라디언트 부스팅은 다음과 같은 계보를 그리며 발전해 왔습니다.

```
[1999/2001] GBM (오리지널 이론 정립, Leo Breiman / Jerome Friedman)
       │
[2014년] XGBoost (병렬 연산 도입, GBM의 대중화 및 속도 혁신)
       │
[2016년] LightGBM (Leaf-wise, GOSS, EFB 도입으로 대용량 데이터 초고속 처리)
       │
[2017년] CatBoost (범주형 변수 처리 최적화, 과적합 방지 최강자)

```

1. **GBM (1999~2001):** 통계학자 제롬 프리드먼 등에 의해 정립되었습니다. 이론은 훌륭했으나 하드웨어 한계로 연산 속도가 너무 느렸습니다.
2. **XGBoost (2014):** 시스템 최적화, 병렬 처리, Regularization(규제) 항을 손실 함수에 도입하여 Kaggle 등 데이터 사이언스 대회를 휩쓸었습니다.
3. **LightGBM (2016):** XGBoost보다 수 배 빠른 속도와 적은 메모리 사용량으로 빅데이터 시대의 필수 알고리즘이 되었습니다.
4. **CatBoost (2017):** 러시아의 Yandex가 개발한 알고리즘으로, **Categorical Boosting**의 약자입니다. 텍스트나 범주형(Categorical) 변수가 많을 때 전처리 없이도 엄청난 성능을 내며, 오버피팅을 방지하는 구조(Ordered Boosting)로 짜여 있어 하이퍼파라미터 튜닝 없이도 기본 성능이 매우 뛰어납니다.

---

## 4. 관련 최신 알고리즘 근황 (2020년대 중반 ~ 현재)

딥러닝(Transformer 계열)이 이미지와 자연어 처리를 지배하고 있는 현재도, **정형 데이터(Excel, DB 형태) 영역에서만큼은 여전히 📝 Tree 기반 부스팅 모델이 지배적인 위치**를 차지하고 있습니다. 최근의 근황과 트렌드는 다음과 같습니다.

### ① GPU 가속의 고도화 (RAPIDS 환경 등)

최근 LightGBM과 XGBoost, CatBoost는 모두 NVIDIA GPU 가속을 완벽하게 지원합니다. 특히 NVIDIA의 **RAPIDS cuDF** 라이브러리와 연동되어, 과거 CPU로 몇 시간씩 걸리던 수천만 행의 대규모 데이터 부스팅 학습이 현재는 GPU를 통해 **몇 초~몇 분 만에** 끝나는 수준으로 진화했습니다.

### ② "정형 데이터에서도 딥러닝이 GBM을 이길 수 있는가?" 논쟁

최근 수년간 **TabNet, SAINT, FT-Transformer** 등 정형 데이터를 타깃으로 한 딥러닝 모델들이 대거 등장했습니다. 하지만 2022~2024년 발표된 여러 벤치마크 논문(예: *"Why do tree-based models still outperform deep learning on tabular data?"*)에 따르면, 여전히 다음과 같은 이유로 **LightGBM/XGBoost/CatBoost의 판정승**으로 결론이 나고 있습니다.

* **데이터 효율성:** 딥러닝은 거대한 데이터가 필요하지만, GBM은 비교적 작은 데이터셋에서도 강력합니다.
* **유연성:** 의미 없는 피처(Noisy feature)가 섞여 있을 때 딥러닝은 쉽게 무너지지만, 트리 모델은 이를 잘 무시합니다.
* **비용:** 부스팅 모델이 학습 및 추론 비용(비용 및 시간) 측면에서 비교가 안 될 정도로 효율적입니다.

### ③ 하이브리드 및 앙상블 프레임워크의 대세화

요즘 현업에서는 단일 LightGBM만 쓰기보다는, AutoML 프레임워크(예: AutoGluon, FLAML)를 사용하여 LightGBM, CatBoost, XGBoost를 적절히 섞고(Stacking/Ensemble), 이를 딥러닝 모델과 결합하는 하이브리드 방식이 표준으로 자리 잡았습니다.

특히 **AutoGluon** 같은 도구는 복잡한 튜닝 없이도 LightGBM의 성능을 극한으로 끌어올려 주어 엔지니어들의 생산성을 크게 높여주고 있습니다.

---